# BBM 409 - Programming Assignment 2

**PART 1:** Binary Classification with SVM (30 points)  
**PART 2:** Multiclass Classification (70 points)

### Mustafa Ege 2210356088



## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, normalize #normalize added only for better confusion matrix
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from xgboost import XGBClassifier
from ucimlrepo import fetch_ucirepo
import warnings
warnings.filterwarnings('ignore')

---
# PART 1: BINARY CLASSIFICATION WITH SVM (30 POINTS)
---

## Load the Sonar Dataset from UCI Repository

In [ ]:
print("Loading Sonar dataset...")
sonar = fetch_ucirepo(id=151)
X_sonar = sonar.data.features
y_sonar = sonar.data.targets

In [ ]:
print(X_sonar.info())
print("\n\nX_sonar (features):")
print(X_sonar.head())
print("\ny_sonar (target):")
print(y_sonar.head())

## Encode the Target Labels using LabelEncoder()

In [ ]:
le = LabelEncoder()
y_sonar_encoded = le.fit_transform(y_sonar.iloc[:, 0])

print(f"Original classes: {le.classes_}")
print(f"Encoded classes: {np.unique(y_sonar_encoded)}")

## Split the Data into 80% Training and 20% Testing Sets

In [ ]:
X_train_sonar, X_test_sonar, y_train_sonar, y_test_sonar = train_test_split(
    X_sonar, y_sonar_encoded, test_size=0.2, random_state=17, stratify=y_sonar_encoded
) #stratify helps us making the proportion of classes  in training and test sets roughly the same

print(f"X_train shape: {X_train_sonar.shape}")
print(f"X_test shape: {X_test_sonar.shape}")

---
## 1.1. Linear Kernel SVM without Hyperparameter Tuning (5 points)
---

## Create a Pipeline with StandardScaler and Linear SVM

In [ ]:
linear_svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear', random_state=17))
])

In [ ]:
hardmargin_linear_svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    # Set C to a very large number to minimize the slack
    ('svm', SVC(kernel='linear', C=1e6, random_state=17)) 
])

## Train the Linear SVM Model

In [ ]:
print("Training Linear SVM...")
linear_svm_pipeline.fit(X_train_sonar, y_train_sonar)
hardmargin_linear_svm_pipeline.fit(X_train_sonar, y_train_sonar)
print("Training complete.")

## Predict on the Test Set and Calculate Accuracies and Print it

In [ ]:
y_train_pred_linear = linear_svm_pipeline.predict(X_train_sonar)
y_test_pred_linear = linear_svm_pipeline.predict(X_test_sonar)

y_train_pred_hardmargin = hardmargin_linear_svm_pipeline.predict(X_train_sonar)
y_test_pred_hardmargin = hardmargin_linear_svm_pipeline.predict(X_test_sonar)


In [ ]:
train_accuracy_linear = accuracy_score(y_train_sonar, y_train_pred_linear)
test_accuracy_linear = accuracy_score(y_test_sonar, y_test_pred_linear)

train_accuracy_hardmargin = accuracy_score(y_train_sonar, y_train_pred_hardmargin)
test_accuracy_hardmargin = accuracy_score(y_test_sonar, y_test_pred_hardmargin)


In [ ]:
print(f"Linear SVM Train Accuracy: {train_accuracy_linear:.4f}")
print(f"Linear SVM Test Accuracy: {test_accuracy_linear:.4f}")

print(f"Hardmargin Linear SVM Train Accuracy: {train_accuracy_hardmargin:.4f}")
print(f"Hardmargin Linear SVM Test Accuracy: {test_accuracy_hardmargin:.4f}")

## Display Classification Report for Linear SVM

In [ ]:
linear_svm_results = {
    'train_accuracy': train_accuracy_linear,
    'test_accuracy': test_accuracy_linear,
    'report': classification_report(y_test_sonar, y_test_pred_linear, target_names=le.classes_, output_dict=True)
}
print(f"Test Classification Report:\n{classification_report(y_test_sonar, y_test_pred_linear, target_names=le.classes_)}")

## Display Confusion Matrix for Linear SVM

In [ ]:
cm_linear = confusion_matrix(y_test_sonar, y_test_pred_linear)

plt.figure(figsize=(8, 5))
sns.heatmap(cm_linear, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix - Linear SVM')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

---
## 1.2. SVM with GridSearchCV and 5-Fold Cross-Validation (15 points)
---

## Create a Pipeline for GridSearchCV

In [ ]:
tuned_svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(random_state=17))
])

## Define the Parameter Grid for Different Kernels

In [ ]:
C_values = [0.01, 0.1, 1, 10, 100, 1000]
gamma_values = ['scale', 'auto', 0.001, 0.01, 0.1]
degree_values = [2, 3, 4, 5]

In [ ]:
# Parameter grid for Linear Kernel
param_grid_linear = {
    'svm__kernel': ['linear'],
    'svm__C': C_values,
}

# Parameter grid for RBF Kernel
param_grid_rbf = {
    'svm__kernel': ['rbf'],
    'svm__C': C_values,
    'svm__gamma': gamma_values
}

# Parameter grid for Polynomial Kernel
param_grid_poly = {
    'svm__kernel': ['poly'],
    'svm__C': C_values,
    'svm__gamma': gamma_values,
    'svm__degree': degree_values
}

# Combine all parameter grids
param_grid = [param_grid_linear, param_grid_rbf, param_grid_poly]

# Display the total number of combinations to be checked
total_combinations = len(C_values) + (len(C_values) * len(gamma_values)) + (len(C_values) * len(gamma_values) * len(degree_values))
print(f"Total number of parameter combinations to check: {total_combinations}")

## Create Stratified 5-Fold Cross-Validation

In [ ]:
cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=17)

## Initialize and Run GridSearchCV on Training Data

In [ ]:
grid_search_svm = GridSearchCV(
    estimator=tuned_svm_pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=cv_stratified,
    verbose=3,
    n_jobs=-1 
)

grid_search_svm.fit(X_train_sonar, y_train_sonar)

## Display Best Parameters and Cross-Validation Score

In [ ]:
# Store the best estimator and result
best_svm_model = grid_search_svm.best_estimator_
tuned_svm_results = grid_search_svm.cv_results_

# Display best parameter and cross validation score
print("--- Best Model Summary ---")
print(f"Best Cross-Validation Score: {grid_search_svm.best_score_:.4f}")
print(f"Best Parameters: {grid_search_svm.best_params_}")

## Evaluate the Best Model on the Test Set

In [ ]:
y_test_pred_tuned = best_svm_model.predict(X_test_sonar)
test_accuracy_tuned = accuracy_score(y_test_sonar, y_test_pred_tuned)
print(f"Tuned SVM Test Accuracy (on unseen data): {test_accuracy_tuned:.4f}")

## Display Top 5 Parameter Combinations

In [ ]:
sorted_indices = tuned_svm_results['mean_test_score'].argsort()[::-1][:20]

top_20_results = pd.DataFrame({
    'Rank': np.arange(1, 21),
    'Mean CV Score': np.array(tuned_svm_results['mean_test_score'])[sorted_indices],
    'Parameters': np.array(tuned_svm_results['params'])[sorted_indices],
})

print("\nTop 20 Parameter Combinations:")
pd.set_option('display.max_colwidth', None)
display(top_20_results)

## Display Classification Report for Tuned SVM

In [ ]:
tuned_svm_report = classification_report(y_test_sonar, y_test_pred_tuned, target_names=le.classes_, output_dict=True)
print("\nTuned SVM Classification Report (Test Set):")
print(classification_report(y_test_sonar, y_test_pred_tuned, target_names=le.classes_))

## Display Confusion Matrix for Tuned SVM

In [ ]:
cm_tuned = confusion_matrix(y_test_sonar, y_test_pred_tuned)

plt.figure(figsize=(8, 5))
sns.heatmap(cm_tuned, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix - Tuned SVM (Best Model)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## Compare Linear SVM vs Tuned SVM Results. Compare experimental results. Explain the performance impact of linear and nonlinear kernels. Why is kernel trick important? Also explain why using k-fold cross validation is more advantageous than train-test split. (10 points)

In [ ]:
best_params = grid_search_svm.best_params_
best_cv_score = grid_search_svm.best_score_
best_kernel = best_params.get('svm__kernel', 'N/A')

# Get linear SVM results
linear_test_acc = linear_svm_results['test_accuracy']

print("--- Performance Comparison (Test Set) ---")

comparison_data = {
    'Model': ['Linear SVM (Untuned)', f'Tuned SVM ({best_kernel.upper()} Kernel)'],
    'Test Accuracy': [f"{linear_test_acc:.4f}", f"{test_accuracy_tuned:.4f}"],
    'Best CV Score': ['N/A', f"{best_cv_score:.4f}"],
    'Best Hyperparameters': ['C=1.0 (Default), Kernel=linear', str(best_params)]
}

comparison_df = pd.DataFrame(comparison_data)

print(comparison_df.to_markdown(index=False))

# Store final test accuracies for Part 2 comparison
linear_svm_test_accuracy = comparison_df.iloc[0]['Test Accuracy']
tuned_svm_test_accuracy = comparison_df.iloc[1]['Test Accuracy']

### Soft Margin and Hard Margin SVM

We first trained both with **soft margin** and **hard margin** SVMs. Even though hard margin SVM reached 100% accuracy on the training set, it showed worse results in the test set compared to soft margin SVM. 
This shows that training data is linearly separable and hard margin SVM achieved to find the hyperplane that separates the classes. However, when it faced the test data which is not separable as seen from the results, accuracy dropped dramatically. 

In other words we get low bias since we fit the data perfectly, but high variance meaning less generalization with the hard margin, basically overfitting. So we will continue with soft margin SVM. 

### Linear SVM vs Tuned SVM Results

We have already seen that the data is not linearly separable but we can also see it from the tuned model SVM achieving significantly higher test accuracy 0.9048 compared to the Untuned Linear SVM 0.83.

Tuned SVM with RBF kernel achieved higher test accuracy (0.90) than linear svm model (0.83). 

The best model used the RBF (Radial Basis Function) kernel and found the best hyperparameters using GridSearchCV. The RBF kernel, which is not linear, let the SVM implicitly map the data into a higher-dimensional space where a linear boundary could be found. This made the SVM work better at generalizing. The fact that the top CV results all have the same score of 0.8431 shows that the parameters chosen are strong.

### Kernel Trick

The difference in performance shows how important kernels are. The Linear kernel can only fit a flat hyperplane, which doesn't work when the data points are all mixed up in a complicated way. The nonlinear RBF kernel, on the other hand, changes the input features into a space with many more dimensions where a linear separation is possible.

The Kernel Trick makes this transformation possible. The kernel trick gives us a shortcut: it calculates the dot product between the transformed vectors in high-dimensional space using only the original input features. This is done instead of actually calculating and storing the new, high-dimensional feature vectors, which might not be possible. This lets us deal with complicated nonlinearity without using a lot of memory and processing power, which makes algorithms like SVM useful for real-world nonlinear problems.

### Advantage of K-Fold Cross-Validation
Using a simple train-test split for hyperparameter selection would have been unreliable, especially with a dataset of only 208 samples. The result would have been highly dependent on the accidental makeup of that single 20% test set.

K-Fold Cross-Validation (with 5 splits and stratification in my case) overcomes this instability. By iteratively training the model on 80% of the data and validating on the remaining 20%, and repeating this five times, we ensure that every single data point is used both for training and validation. The final average cross-validation score of 0.8431 for the best RBF model provides a far more robust and stable estimate of the model's true generalization ability than a single split could, giving me high confidence in selecting that set of optimal hyperparameters.

---
# PART 2: MULTICLASS CLASSIFICATION (70 POINTS)
---

## Load the Dry Beans Dataset from UCI Repository

In [ ]:
print("Loading Dry Beans dataset...")
dry_bean = fetch_ucirepo(id=602)
X_beans = dry_bean.data.features
y_beans = dry_bean.data.targets

## Encode the Target Labels using LabelEncoder

In [ ]:
le_beans = LabelEncoder()
y_beans_encoded = le_beans.fit_transform(y_beans.iloc[:, 0])

print(f"Original Bean Classes: {le_beans.classes_}")
print(f"Encoded Bean Classes: {np.unique(y_beans_encoded)}")

## Split the Data into 80% Training and 20% Testing Sets

In [ ]:
X_train_beans, X_test_beans, y_train_beans, y_test_beans = train_test_split(
    X_beans, y_beans_encoded, test_size=0.2, random_state=17, stratify=y_beans_encoded
)

print(f"X_train_beans shape: {X_train_beans.shape}")
print(f"X_test_beans shape: {X_test_beans.shape}")

## Scale the Features using StandardScaler

In [ ]:
scaler_beans = StandardScaler()
X_train_scaled = scaler_beans.fit_transform(X_train_beans)
X_test_scaled = scaler_beans.transform(X_test_beans)

---
## 2.1. Multinomial Logistic Regression (20 points)
---

## Define the Multinomial Logistic Regression Class

In [ ]:
class MultinomialLogisticRegression():
    def __init__(self, learning_rate=0.01, epochs=500, reg_lambda=0.0):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.reg_lambda = reg_lambda
        self.W = None
        self.b = None
        self.K = None  # Number of classes
        self.N = None  # Number of samples

    def _softmax(self, z):
        """Computes the softmax probabilities."""
        # Subtract max(z) for numerical stability
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def _cross_entropy(self, X, y_one_hot, P):
        """Computes the cross-entropy loss with L2 regularization."""
        # Cross-Entropy Loss
        log_likelihood = -np.log(P[range(self.N), np.argmax(y_one_hot, axis=1)])
        loss = np.sum(log_likelihood) / self.N
        
        # L2 Regularization (on weights W only, excluding bias b)
        reg_loss = (self.reg_lambda / (2 * self.N)) * np.sum(self.W**2)
        
        return loss + reg_loss

    def fit(self, X, y):
        self.N, M = X.shape  # N=samples, M=features
        self.K = len(np.unique(y))  # Number of classes
        
        # Initialize weights (W) and bias (b)
        self.W = np.zeros((M, self.K))
        self.b = np.zeros(self.K)
        
        # Convert y to one-hot encoding for easier computation of loss and gradient
        y_one_hot = np.zeros((self.N, self.K))
        y_one_hot[np.arange(self.N), y] = 1
        
        # Gradient Descent
        for epoch in range(self.epochs):
            # 1. Linear combination: X * W + b
            z = X @ self.W + self.b
            
            # 2. Softmax probabilities
            P = self._softmax(z)
            
            # 3. Gradient computation (Error = P - y_one_hot)
            error = P - y_one_hot
            
            # Gradient for weights (dW) and bias (db)
            dW = (X.T @ error) / self.N
            db = np.sum(error, axis=0) / self.N
            
            # Add L2 Regularization term to weight gradient
            dW += (self.reg_lambda / self.N) * self.W
            
            # 4. Update weights and bias
            self.W -= self.learning_rate * dW
            self.b -= self.learning_rate * db
            
            # Check loss periodically 
            if (epoch + 1) % 100 == 0:
                loss = self._cross_entropy(X, y_one_hot, P)
                print(f"Epoch {epoch + 1}/{self.epochs}, Loss: {loss:.4f}")

    def predict_proba(self, X):
        z = X @ self.W + self.b
        return self._softmax(z)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

    def score(self, X, y):
        y_pred = self.predict(X)
        return accuracy_score(y, y_pred)

    # Methods required for GridSearchCV compatibility
    def get_params(self, deep=True):
        return {"learning_rate": self.learning_rate, "epochs": self.epochs, "reg_lambda": self.reg_lambda}

    def set_params(self, **parameters):
        for parameter, value in parameters.items():
            setattr(self, parameter, value)
        return self

## Define Hyperparameter Grid for Multinomial Logistic Regression

In [ ]:
## Define Hyperparameter Grid for Multinomial Logistic Regression
param_grid_mlr = {
    'learning_rate': [0.01, 0.05, 0.1],
    'epochs': [200, 300, 500],
    'reg_lambda': [0.0, 0.01, 0.1]
}

# Initialize the model and GridSearchCV
mlr_model = MultinomialLogisticRegression()

cv_stratified_beans = StratifiedKFold(n_splits=5, shuffle=True, random_state=17)

print("Starting GridSearchCV for Multinomial Logistic Regression...")
grid_search_mlr = GridSearchCV(
    estimator=mlr_model,
    param_grid=param_grid_mlr,
    scoring='accuracy',
    cv=cv_stratified_beans,
    verbose=1,
    n_jobs=-1
)

## Run GridSearchCV for Multinomial Logistic Regression

In [ ]:
grid_search_mlr.fit(X_train_scaled, y_train_beans)
print("GridSearchCV complete.")

# Store the best estimator
best_mlr_model = grid_search_mlr.best_estimator_

## Evaluate Multinomial Logistic Regression on Test Set

In [ ]:
print("--- Multinomial Logistic Regression Results ---")
print(f"Best CV Score: {grid_search_mlr.best_score_:.4f}")
print(f"Best Parameters: {grid_search_mlr.best_params_}")

y_test_pred_mlr = best_mlr_model.predict(X_test_scaled)
test_accuracy_mlr = accuracy_score(y_test_beans, y_test_pred_mlr)
print(f"Test Accuracy: {test_accuracy_mlr:.4f}")

# Store results for later comparison in Part 2.4
mlr_test_accuracy = test_accuracy_mlr
mlr_best_params = grid_search_mlr.best_params_
mlr_best_cv_score = grid_search_mlr.best_score_

## Display Classification Report for Multinomial Logistic Regression

In [ ]:
target_names_beans = le_beans.classes_
print("\nClassification Report:")
print(classification_report(y_test_beans, y_test_pred_mlr, target_names=target_names_beans))

## Display Confusion Matrix for Multinomial Logistic Regression

In [ ]:
cm_mlr = confusion_matrix(y_test_beans, y_test_pred_mlr)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_mlr, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names_beans, yticklabels=target_names_beans)
plt.title('Confusion Matrix - Multinomial Logistic Regression')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

Standard confusion matrix shows raw counts but normalized confusion matrix (where each row sums to 1.0) shows much better the misclassification rate per true class. (normalize from sklearn was imported only for this reason and the next confusion matrices for decision tree and xgboost.)

In [ ]:
cm_mlr_normalized = normalize(cm_mlr, axis=1, norm='l1')

plt.figure(figsize=(8,6))
sns.heatmap(cm_mlr_normalized, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=target_names_beans, yticklabels=target_names_beans, cbar=True)
plt.title('Normalized Confusion Matrix (MLR) - Row Sums to 1.0')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig('mlr_normalized_confusion_matrix.png')
plt.show()

## Feature Importance Plot 

To see which features are prioritized for the model

In [ ]:
W_matrix = best_mlr_model.W
feature_names = X_beans.columns

abs_magnitude = np.mean(np.abs(W_matrix), axis=1)
importance_mlr = pd.Series(abs_magnitude, index=feature_names)
top_10_importance_mlr = importance_mlr.nlargest(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_10_importance_mlr.values, y=top_10_importance_mlr.index, palette='magma')
plt.title('Top 10 Feature Influence (MLR - Avg Absolute Coefficients)')
plt.xlabel('Average Absolute Coefficient Magnitude')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('mlr_feature_influence.png')
plt.show()

---
## 2.2. Decision Tree (20 points)
---

###  Define the Decision Tree model. Use GridSearchCV to find the best hyperparameters. Perform multiclass classification on the Dry Beans dataset. Print best hyperparameters, classification report and confusion matrix. (15 points)

## Define Hyperparameter Grid for Decision Tree

In [ ]:
param_grid_dt = {
    'max_depth': [10, 15, 20, 25, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

# Initialize the Decision Tree model
dt_model = DecisionTreeClassifier(random_state=17)

cv_stratified_beans = StratifiedKFold(n_splits=5, shuffle=True, random_state=17)

## Run GridSearchCV for Decision Tree

In [ ]:
print("Starting GridSearchCV for Decision Tree...")
grid_search_dt = GridSearchCV(
    estimator=dt_model,
    param_grid=param_grid_dt,
    scoring='accuracy',
    cv=cv_stratified_beans,
    verbose=1,
    n_jobs=-1
)

grid_search_dt.fit(X_train_scaled, y_train_beans)
print("GridSearchCV complete.")

best_dt_model = grid_search_dt.best_estimator_
y_test_pred_dt = best_dt_model.predict(X_test_scaled)
test_accuracy_dt = accuracy_score(y_test_beans, y_test_pred_dt)

# Store results for later comparison in Part 2.4
dt_test_accuracy = test_accuracy_dt
dt_best_params = grid_search_dt.best_params_
dt_best_cv_score = grid_search_dt.best_score_

print("--- Decision Tree Results ---")
print(f"Best CV Score: {dt_best_cv_score:.4f}")
print(f"Best Parameters: {dt_best_params}")
print(f"Test Accuracy: {dt_test_accuracy:.4f}")


## Display Classification Report for Decision Tree

In [ ]:
## Display Classification Report for Decision Tree
target_names_beans = le_beans.classes_
print("\nClassification Report:")
print(classification_report(y_test_beans, y_test_pred_dt, target_names=target_names_beans))

## Display Confusion Matrix for Decision Tree

In [ ]:
cm_dt = confusion_matrix(y_test_beans, y_test_pred_dt)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names_beans, yticklabels=target_names_beans)
plt.title('Confusion Matrix - Decision Tree (Tuned)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
cm_dt_normalized = normalize(cm_dt, axis=1, norm='l1')

plt.figure(figsize=(8,6))
sns.heatmap(cm_dt_normalized, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=target_names_beans, yticklabels=target_names_beans, cbar=True)
plt.title('Normalized Confusion Matrix (Decision Tree) - Row Sums to 1.0')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig('dt_normalized_confusion_matrix.png')
plt.show()

## Feature Importance Plot 

To see which features are prioritized for the model

In [ ]:
importance_dt = pd.Series(best_dt_model.feature_importances_, index=X_beans.columns)
top_10_importance_dt = importance_dt.nlargest(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_10_importance_dt.values, y=top_10_importance_dt.index, palette='crest')
plt.title('Top 10 Feature Importance (Decision Tree)')
plt.xlabel('Feature Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('dt_feature_importance.png')
plt.show()

###  Describe how a decision tree builds its decision structure. (5 points)

A decision tree uses greedy process to find separate the data based on its features.

In every step it tries to maximize **information gain**.

The process is as follows:

1. Start at the root node where the root node is the entire training set.

2. At the start node and every following node, examine every possible feature and every possible value for that feature as a potential split point.

    Evaluate the split based on information gain. In other words we can say it chooses the feature in which we understand more stuff about the output.
    For this, mainly we use entropy. 

    Also this is where the algorithm is greedy, since it chooses the feature with most information gain at the present node, not thinking further.

3. The dataset is divided into two or more subsets based on the chosen split, and the process is repeated recursively on each new child node.

4. This process continues until a stopping criterion is met.
    
    Stopping criterion might be many things,

    * The node contains samples all belonging to a single class.
    * The tree reaches the maximum allowed depth.
    * The number of samples in a node is less than the minimum samples required to split.
    * The number of samples in a new leaf node would be less than the minimum samples required for a leaf.

Finally, it stops and we reach a point where there are leaf nodes. And any sample that reaches a leaf node is assigned that leaf node's class.



---
## 2.3. XGBoost (20 points)
---

###  Define the XGBoost model. Use GridSearchCV to find the best hyperparameters. Perform multiclass classification on the Dry Beans dataset. Print best hyperparameters, classification report and confusion matrix(10 points)

## Define Hyperparameter Grid for XGBoost

In [ ]:
param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 7, 10],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Initialize the XGBoost model
xgb_model = XGBClassifier(
    objective='multi:softmax', 
    eval_metric='mlogloss', 
    use_label_encoder=False, 
    random_state=17
)

cv_stratified_beans = StratifiedKFold(n_splits=5, shuffle=True, random_state=17)

## Run GridSearchCV for XGBoost

In [ ]:
print("Starting GridSearchCV for XGBoost..")
grid_search_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid_xgb,
    scoring='accuracy',
    cv=cv_stratified_beans,
    verbose=1,
    n_jobs=-1
)

grid_search_xgb.fit(X_train_scaled, y_train_beans)
print("GridSearchCV complete.")

best_xgb_model = grid_search_xgb.best_estimator_
y_test_pred_xgb = best_xgb_model.predict(X_test_scaled)
test_accuracy_xgb = accuracy_score(y_test_beans, y_test_pred_xgb)

# Store results for later comparison in Part 2.4
xgb_test_accuracy = test_accuracy_xgb
xgb_best_params = grid_search_xgb.best_params_
xgb_best_cv_score = grid_search_xgb.best_score_

## Evaluate XGBoost on Test Set

In [ ]:
print("--- XGBoost Results ---")
print(f"Best CV Score: {xgb_best_cv_score:.4f}")
print(f"Best Parameters: {xgb_best_params}")
print(f"Test Accuracy: {xgb_test_accuracy:.4f}")

## Display Classification Report for XGBoost

In [ ]:
target_names_beans = le_beans.classes_
print("\nClassification Report:")
print(classification_report(y_test_beans, y_test_pred_xgb, target_names=target_names_beans))

## Display Confusion Matrix for XGBoost

In [ ]:
cm_xgb = confusion_matrix(y_test_beans, y_test_pred_xgb)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names_beans, yticklabels=target_names_beans)
plt.title('Confusion Matrix - XGBoost (Tuned)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
cm_normalized = normalize(cm_xgb, axis=1, norm='l1')

plt.figure(figsize=(8,6))
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=target_names_beans, yticklabels=target_names_beans, cbar=True)
plt.title('Normalized Confusion Matrix (XGBoost) - Row Sums to 1.0')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig('xgb_normalized_confusion_matrix.png')
plt.show()

## Feature Importance Plot 

To see which features are prioritized for the model

In [ ]:
importance = pd.Series(best_xgb_model.feature_importances_, index=X_beans.columns)
top_10_importance = importance.nlargest(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_10_importance.values, y=top_10_importance.index, palette='viridis')
plt.title('Top 10 Feature Importance (XGBoost)')
plt.xlabel('Feature Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('xgb_feature_importance.png')
plt.show()

### What are the revolutionary features of XGBoost compared to other tree-based models? (10 points)

* **Gradient Boosting with Second-Order Optimization**

    Unlike classical boosting methods that use only gradients, XGBoost uses both the first and second derivatives. This leads to more accurate split decisions and faster convergence.

* **Regularization (L1 & L2)**

    It adds regularization to control model complexity which reduces overfitting. This controls overfitting more effectively than earlier gradient boosting implementations.

* **System Optimization for Speed**

    XGBoost is pretty efficient with 2 features. 
    * Even though boosting is sequential, it parallelizes the search for the best split.
    * It optimizes memory access patterns with cache, making it perform better on large datasets.

* **Handling Missing Values Automatically**
    
    It natively handles missing values and sparse data. The algorithm automatically learns the optimal direction for “missing” values during training and it improves performance.

* **Approximate Split Finding for Large Datasets**
    
    When datasets are too large to evaluate every possible split, it uses an approximate greedy algorithm based on percentiles. This preserves accuracy while reducing computation.

* **Out-of-Core Computing**
    
    It can handle datasets larger than memory using disk spill-over and block compression, making it suitable for huge datasets.

* **Pruning with Max Depth**
    
    XGBoost uses a depth-first tree pruning strategy instead of traditional level-wise growth. This produces more optimized and compact trees.

##  2.4. Compare the classification results of your Multinomial Logistic Regression, Decision Tree, and XGBoost models on the Dry Beans dataset. Discuss the comparison in terms of overall model performance, risk of overfitting, model complexity and the scenarios in which each model is most effective. (10 points)

In [ ]:
# final comparison table
final_comparison_data = {
    'Model': ['Multinomial Logistic Regression (MLR)', 'Decision Tree (DT)', 'XGBoost (XGB)'],
    'Test Accuracy': [mlr_test_accuracy, dt_test_accuracy, xgb_test_accuracy],
    'Best CV Score': [mlr_best_cv_score, dt_best_cv_score, xgb_best_cv_score]
}

final_comparison_df = pd.DataFrame(final_comparison_data)

print("\n--- Final Multiclass Classification Performance Comparison ---")
print(final_comparison_df.to_markdown(index=False))

In [ ]:
plt.figure(figsize=(8, 5))
bars = sns.barplot(x='Model', y='Test Accuracy', data=final_comparison_df, palette='viridis')
plt.title('Final Test Accuracy Comparison on Dry Beans Dataset')
plt.ylim(0.85, 1.0) 
plt.ylabel('Test Accuracy')
plt.xlabel('Classification Model')

# Add accuracy labels on bars
for bar in bars.patches:
    bars.annotate(f'{bar.get_height():.4f}',
                  (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                  ha='center', va='bottom', fontsize=10, xytext=(0, 5),
                  textcoords='offset points')

plt.savefig('multiclass_accuracy_comparison.png')
plt.show()

### Model Performance Analysis

**XGBoost** achieved the highest test accuracy around 0.93, which confirms that it is the top confirming algorithm for complex datasets. Sequential boosting and advanced optimization helped it find the most accurate decision boundary.

The **Multinomial Logistic Regression** performed quite well with around 0.915, even slightly outperforming the single **Decision Tree** with around 0.91 This suggests that the features in the Dry Beans dataset are highly informative, and the class boundaries, while non-linear, might be relatively separable in the high-dimensional space, allowing the linear MLR to capture most of the signal.

### Risk of Overfitting

* **Decision Tree (DT):** Generally carries a higher risk of overfitting. Although tuning helped, its accuracy dropped slightly from its CV score, suggesting minor overfitting or variance compared to the CV folds.
* **XGBoost (XGB):** Shows excellent generalization. Its test accuracy is nearly identical to its robust cross-validation score, indicating that its regularization mechanisms were effective at preventing the ensemble from overfitting.
* **Multinomial Logistic Regression:** Has the lowest risk of overfitting. As a simple linear model with L2 regularization, it is actuallt far from overfitting and more prone to underfitting complex data. Its small gap between CV score and test score is expected.

### Model Complexity

* **XGBoost** is the most complex model. It involves hundreds of interdependent, deep trees, making it a like a black box. But it results in maximum accuracy.
* **Decision Tree** is moderately complex. It offers the best interpretability among tree based models, which mean we understand clearly what feature results in what output as the decision rules can be visualized. However, the optimal tree found by tuning may still be deep and complex.
* **Multinomial Logistic Regression** is the least complex model. It's really transparent, feature importance is directly proportional to the magnitude of its learned weights, making it the most easily interpretable model.

### Scenarios in Which Each Model is Most Effective

* **XGBoost**: When we need the absolute highest accuracy, regardless of how complex the model is. Best for competitive data science or critical industrial tasks (like predicting machine failure) where a small increase in accuracy means large savings.

* **Decision Tree**: When we need a simple, and fast explanation for a prediction. It is great as a quick baseline or when a clear, if then else kind of rules is needed to explain decisions to a non-technical audience.

* **Multinomial Logistic Regression**: When the problem is simple, or model transparency is essential. Perfect for creating a quick, understandable baseline, or in social sciences/econometrics where we need to interpret the exact influence of each feature for example, 'How much does one extra year of education increase the odds of a certain outcome?'.